> Repair checkpoint (2026-11-01): the approved paper governs this source.
> ALL stored execution counts, outputs and numerical result narratives below
> are historical/stale relative to the repaired code; no cells were executed.
> Historical email counts include a different source pool and are not current
> expectations. No claim is made that the historical holdout is untouched.
> New outputs use an isolated run namespace and exclusive file creation.
> P 4.2.1/Tables 4.1, 4.2 and 4.7: PhiUSIIL and UCI are technical U1
> branches. They provide no content-dependent SE-BRL contribution here.
> The approved content-bearing Web dataset is not substituted by PhiUSIIL.
> Notebook 05 remains a historical consumer and is not compatible with this
> new namespace/canonical feature dictionary until separately migrated.

# SE-BRL Feature Extraction

## Purpose

This notebook derives the modality-aware Social Engineering Behavior
Representation Layer (SE-BRL) from the fixed preprocessing outputs produced by
`03_preprocessing.ipynb`.

The SE-BRL representation consists of four behavioral dimensions supported by
twelve subordinate indicators:

### Pressure and Threat
- Urgency
- Fear or Threat
- Scarcity

### Lure and Attention
- Curiosity
- Reward or Lure

### Trust and Identity
- Authority
- Impersonation
- Brand Exploitation
- Confidentiality or Isolation

### Requested Action and Consequence
- Call-to-Action
- Credential or Sensitive-Data Request
- Financial-Action Request

Feature extraction is deterministic and does not introduce researcher-assigned
behavior labels.

Each behavioral dimension is represented using one of three evidence states:

- `Supported` — qualifying evidence is present;
- `Absent` — the dimension is assessable but qualifying evidence is not found;
- `Unavailable` — the required modality or evidence field is not assessable.

Exact evidence references are retained separately for explainability and
validation but are not directly used as predictive-vector inputs.

## Modality Rules

Email observations may be assessed using available subject/body content,
sender or identity relations, links, formatting, and related message fields.

Webpage observations may be assessed using available text, HTML/DOM structure,
forms, links, brand/domain relations, and other supplied page evidence.

Standalone URL or engineered URL-feature observations do not provide sufficient
content for language-dependent SE-BRL behavioral inference. Their corresponding
behavioral dimensions are therefore marked `Unavailable` rather than `Absent`.

## Inputs

Fixed preprocessing outputs:

`experiments/data/processed/preprocessed/`

Partition metadata:

`experiments/results/preprocessing/partitions/`

## Outputs

The notebook will produce:

- deterministic indicator-level evidence;
- dimension-level SE-BRL states;
- SE-BRL behavioral vectors;
- modality availability masks; and
- evidence records for downstream validation and explanation.

The fixed observation, group, partition, and cross-validation assignments from
`03_preprocessing.ipynb` are preserved unchanged.
**Development-only preparation:** after validating supplied population and
partition metadata, only development observations enter feature processing.
Calibration/test observations do not enter lexical or structural callbacks.
New feature files therefore cover development IDs only. This is not a final
calibration/test export and does not reseal the exposed historical holdout.

**Explicit branch selection:** set SELECTED_BRANCHES to a nonempty tuple of
"Email", "PhiUSIIL" and/or "UCI". The default retains the combined workflow.
U1-only selections neither load Email inputs nor run Email lexical examples,
rules or its freeze gate. All retained outputs below remain historical/stale.


## 1. Environment and Paths

Feature extraction operates only on the fixed preprocessing outputs.

No partition assignments are regenerated and no target labels are modified in
this notebook.

In [ ]:
from pathlib import Path
import hashlib
import json
import re

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 120)

In [ ]:
cwd = Path.cwd().resolve()

REPO_ROOT = next(
    (
        p for p in [cwd, *cwd.parents]
        if (p / ".git").exists()
        and (p / "experiments").exists()
    ),
    None
)

if REPO_ROOT is None:
    raise RuntimeError("Could not locate repository root.")

import sys
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from ml.notebook_contracts import (
    PREPARATION_RUN_ID, CONTRACT_VERSION, DATASET_ROLES,
    canonical_notebook_codebook, technical_indicator_columns, validate_feature_boundary,
    validate_metadata_alignment, email_evidence_availability, assert_new_outputs,
)

def as_records(df):
    return df.astype(object).where(pd.notna(df), None).to_dict("records")

from ml.branch_preparation import (
    ALL_BRANCHES, select_branches, load_selected_inputs,
    prepare_selected_branches, export_selected_handoff, SHARED_OUTPUTS,
)
SELECTED_BRANCHES = select_branches(ALL_BRANCHES)  # e.g. ("PhiUSIIL", "UCI")
RUN_ID = PREPARATION_RUN_ID  # Use a fresh ID for a separate preparation; D9 must use the same ID.
RUN_ROOT = REPO_ROOT / "experiments" / "runs" / RUN_ID
# No scientific rule, language-coverage or partial-availability choice is made.
FROZEN_RULE_SPECIFICATION_REFERENCE = None
PREPROCESSED_DIR = (
    RUN_ROOT
    / "data"
    / "processed"
    / "preprocessed"
)

PREPROCESSING_RESULTS_DIR = (
    RUN_ROOT
    / "results"
    / "preprocessing"
)

FEATURE_RESULTS_DIR = (
    RUN_ROOT
    / "results"
    / "feature_extraction"
)

FEATURE_DATA_DIR = (
    RUN_ROOT
    / "data"
    / "processed"
    / "features"
)

# Directories are created only after input/output validation at export.

print("Repository root:", REPO_ROOT)
print("Preprocessed input:", PREPROCESSED_DIR)
print("Preprocessing results:", PREPROCESSING_RESULTS_DIR)
print("Feature results:", FEATURE_RESULTS_DIR)
print("Feature data:", FEATURE_DATA_DIR)


In [ ]:
print("Preprocessed input exists:", PREPROCESSED_DIR.exists())
print("Preprocessing results exist:", PREPROCESSING_RESULTS_DIR.exists())
print("Feature results exist:", FEATURE_RESULTS_DIR.exists())
print("Feature data exists:", FEATURE_DATA_DIR.exists())

## 2. Canonical SE-BRL Codebook

The SE-BRL representation is defined independently of any individual dataset.

Each behavioral indicator belongs to exactly one behavioral dimension, and each
dimension is assigned one of three evidence states:

- `Supported` — qualifying behavioral evidence is detected;
- `Absent` — the dimension is assessable but qualifying evidence is not detected;
- `Unavailable` — the modality does not provide sufficient evidence to assess
  the dimension.

The codebook defined in this section is used consistently across extraction,
validation, and downstream experiments.

In [ ]:
# Canonical identifiers only; no claim that runtime/notebook rules are equivalent.
SEBRL_CODEBOOK = canonical_notebook_codebook()


In [ ]:
codebook_rows = []

for dimension_id, dimension_info in SEBRL_CODEBOOK.items():
    for indicator_id, indicator_name in dimension_info["indicators"].items():
        codebook_rows.append({
            "dimension_id": dimension_id,
            "dimension_name": dimension_info["name"],
            "indicator_id": indicator_id,
            "indicator_name": indicator_name,
        })

sebrl_codebook_df = pd.DataFrame(codebook_rows)

print("Dimensions:", sebrl_codebook_df["dimension_id"].nunique())
print("Indicators:", len(sebrl_codebook_df))

sebrl_codebook_df
sebrl_codebook_df["provenance_contract_version"] = CONTRACT_VERSION
sebrl_codebook_df["provenance_rule_status"] = "not_frozen"


In [ ]:
SEBRL_STATES = [
    "Supported",
    "Absent",
    "Unavailable",
]

print(SEBRL_STATES)

In [ ]:
MODALITY_AVAILABILITY = {
    "email": {dimension: True for dimension in SEBRL_CODEBOOK},
    "webpage": {dimension: True for dimension in SEBRL_CODEBOOK},
    "technical": {dimension: False for dimension in SEBRL_CODEBOOK},
}


In [ ]:
expected_dimensions = set(SEBRL_CODEBOOK.keys())

for modality, availability in MODALITY_AVAILABILITY.items():
    assert set(availability.keys()) == expected_dimensions

print("All modality availability rules cover all SE-BRL dimensions.")

### 2.1 Codebook Validation Result

The canonical SE-BRL codebook contains four behavioral dimensions and twelve
subordinate indicators.

All three supported modalities have explicit availability rules for every
dimension.

Email and webpage observations are behavior-assessable, while standalone URL
feature records are marked unavailable for behavioral inference because they do
not provide message or page-content evidence.

## 3. Load Fixed Preprocessed Inputs

Feature extraction begins from the fixed outputs produced by
`03_preprocessing.ipynb`.

The observation identifiers, leakage-prevention groups, development/calibration/
test partitions, and development cross-validation assignments are treated as
immutable metadata.

This notebook does not regenerate partitions or modify standardized targets.

In [ ]:
from ml.branch_preparation import PREPROCESSED_FILES, PARTITION_FILES
FEATURE_INPUTS = {
    name: PREPROCESSED_DIR / PREPROCESSED_FILES[name]
    for name in SELECTED_BRANCHES
}


In [ ]:
# Only selected input/manifest paths are read; missing selections never fall back.
development_records = load_selected_inputs(
    RUN_ROOT, SELECTED_BRANCHES,
    read_records=lambda path: as_records(pd.read_csv(path)),
)
feature_data = {name: pd.DataFrame(rows) for name, rows in development_records.items()}
for name, df in feature_data.items():
    print(name, len(df), "development rows")


In [ ]:
# The loader validates full population/manifest alignment before retaining development.
for name, df in feature_data.items():
    validate_feature_boundary(as_records(df), dataset=name, development_only=True)


In [ ]:
for name, df in feature_data.items():

    print(f"\n{name}")

    print(
        "Unique observation IDs:",
        df["observation_id"].nunique()
    )

    print(
        "Duplicate observation IDs:",
        df["observation_id"].duplicated().sum()
    )

    print(
        "Missing partitions:",
        df["partition"].isna().sum()
    )

    dev_missing_folds = df.loc[
        df["partition"] == "development",
        "cv_fold"
    ].isna().sum()

    nondev_with_folds = df.loc[
        df["partition"] != "development",
        "cv_fold"
    ].notna().sum()

    print(
        "Development rows missing CV fold:",
        dev_missing_folds
    )

    print(
        "Calibration/test rows with CV fold:",
        nondev_with_folds
    )

## 4. Modality-Specific Evidence Availability

SE-BRL feature extraction depends on the evidence exposed by each modality.

The available fields differ substantially across the email, webpage, and URL
datasets. Behavioral inference is therefore restricted to evidence that is
actually present in the corresponding source representation.

This section records which fields are available for:

- textual content;
- sender or identity information;
- links and URLs;
- domain or brand information;
- structural or form-related webpage evidence; and
- engineered URL-only features.

Fields that are unavailable for a modality are not treated as negative evidence.
They result in `Unavailable` states where the corresponding behavioral
dimension cannot be assessed.

In [ ]:
for name, df in feature_data.items():
    print(f"\n{name}")
    print(df.columns.tolist())

In [ ]:
def has_any_column(df, candidates):
    return any(
        col in df.columns
        for col in candidates
    )

In [ ]:
evidence_availability = []
for name, df in feature_data.items():
    technical = DATASET_ROLES[name].startswith("U1")
    evidence_availability.append({
        "dataset": name, "role": DATASET_ROLES[name],
        "modality": "technical" if technical else "email",
        "text_content": False if technical else has_any_column(df, ["subject", "body"]),
        "sender_identity": False if technical else has_any_column(df, ["sender"]),
        "url_or_link": False if technical else has_any_column(df, ["urls"]),
        "domain_identity": False,
        "structured_web_features": False,
        "engineered_url_features": technical,
    })
evidence_availability_df = pd.DataFrame(evidence_availability)


### 4.1 Paper-defined evidence roles
Only content-bearing Email enters this path's lexical evidence processing.
PhiUSIIL and UCI are technical U1 branches (P Tables 4.1, 4.2 and 4.7):
their title/host/form feature values do not establish behavioral meaning.
Field presence is not row-level assessability; unknown optional links and
identity fields remain unavailable. The content-bearing Web branch remains
unimplemented pending the paper-specified source and its audit.


## 5. Deterministic Indicator Extraction Rules

SE-BRL indicator extraction is rule-based and deterministic.

Each indicator is defined using explicit lexical, structural, or identity-based
evidence that can be reproduced automatically from the available fields.

The extraction rules do not assign behavioral labels manually and do not depend
on researcher judgment at runtime.

Where a modality does not expose the evidence required for an indicator, the
corresponding dimension may be marked `Unavailable` rather than `Absent`.

In [ ]:
SEBRL_RULES = {
    "urgency": {
        "type": "lexical",
        "patterns": [
            r"\burgent\b",
            r"\bimmediately\b",
            r"\basap\b",
            r"\bact now\b",
            r"\bright away\b",
            r"\bwithin \d+ (?:hour|hours|minute|minutes)\b",
            r"\bdeadline\b",
            r"\bexpires?\b",
        ],
    },

    "fear_threat": {
        "type": "lexical",
        "patterns": [
            r"\bsuspended\b",
            r"\bterminated\b",
            r"\blocked\b",
            r"\bunauthorized\b",
            r"\bsecurity alert\b",
            r"\bcompromised\b",
            r"\bpenalty\b",
            r"\blegal action\b",
            r"\baccount closure\b",
        ],
    },

    "scarcity": {
        "type": "lexical",
        "patterns": [
            r"\blimited time\b",
            r"\blimited offer\b",
            r"\bwhile supplies last\b",
            r"\bonly \d+ left\b",
            r"\blast chance\b",
            r"\bexclusive offer\b",
        ],
    },

    "curiosity": {
        "type": "lexical",
        "patterns": [
            r"\bsee what happened\b",
            r"\bcheck this out\b",
            r"\byou won't believe\b",
            r"\bimportant message\b",
            r"\bprivate message\b",
            r"\bsurprise\b",
        ],
    },

    "reward_lure": {
        "type": "lexical",
        "patterns": [
            r"\bwon\b",
            r"\bwinner\b",
            r"\bprize\b",
            r"\breward\b",
            r"\bbonus\b",
            r"\bgift\b",
            r"\bfree\b",
            r"\bcashback\b",
            r"\bclaim\b",
        ],
    },

    "authority": {
        "type": "lexical",
        "patterns": [
            r"\badministrator\b",
            r"\badmin\b",
            r"\bmanagement\b",
            r"\bsecurity team\b",
            r"\bit department\b",
            r"\bgovernment\b",
            r"\bbank\b",
            r"\bmanager\b",
            r"\bceo\b",
        ],
    },

    "impersonation": {
        "type": "identity",
        "patterns": [],
    },

    "brand_exploitation": {
        "type": "identity",
        "patterns": [],
    },

    "confidentiality_isolation": {
        "type": "lexical",
        "patterns": [
            r"\bconfidential\b",
            r"\bdo not share\b",
            r"\bkeep this private\b",
            r"\bsecret\b",
            r"\bdo not tell\b",
            r"\bbetween us\b",
        ],
    },

    "call_to_action": {
        "type": "lexical",
        "patterns": [
            r"\bclick\b",
            r"\bopen\b",
            r"\bvisit\b",
            r"\bdownload\b",
            r"\bverify\b",
            r"\bconfirm\b",
            r"\bupdate\b",
            r"\bsign in\b",
            r"\blog in\b",
            r"\breply\b",
        ],
    },

    "credential_sensitive_data_request": {
        "type": "lexical",
        "patterns": [
            r"\bpassword\b",
            r"\busername\b",
            r"\blogin credentials\b",
            r"\bverification code\b",
            r"\botp\b",
            r"\bpin\b",
            r"\bssn\b",
            r"\bsocial security\b",
            r"\bpersonal information\b",
            r"\baccount details\b",
        ],
    },

    "financial_action_request": {
        "type": "lexical",
        "patterns": [
            r"\btransfer\b",
            r"\bpayment\b",
            r"\bpay\b",
            r"\binvoice\b",
            r"\bwire\b",
            r"\bbank transfer\b",
            r"\bcredit card\b",
            r"\bdebit card\b",
            r"\bfee\b",
            r"\bdeposit\b",
        ],
    },
}


In [ ]:
expected_indicators = set(
    sebrl_codebook_df["indicator_id"]
)

defined_indicators = set(
    SEBRL_RULES.keys()
)

print(
    "Missing rules:",
    expected_indicators - defined_indicators
)

print(
    "Unexpected rules:",
    defined_indicators - expected_indicators
)

assert defined_indicators == expected_indicators

print("All 12 SE-BRL indicators have rule definitions.")

### 5.1 Lexical Matching and Evidence Capture

Lexical indicators are evaluated using deterministic regular-expression rules.

For each indicator, the extraction process records:

- whether qualifying evidence is present; and
- the exact matched text fragments that support the indicator.

Evidence is preserved for explainability and validation. Matched phrases are
not manually edited or reinterpreted.

In [ ]:
def normalize_match_text(value):
    if pd.isna(value):
        return ""

    value = str(value)

    # Normalize whitespace while preserving original wording
    value = re.sub(r"\s+", " ", value)

    return value.strip()


def extract_pattern_evidence(text, patterns):
    text = normalize_match_text(text)

    if not text:
        return []

    evidence = []

    for pattern in patterns:
        matches = re.finditer(
            pattern,
            text,
            flags=re.IGNORECASE
        )

        for match in matches:
            matched_text = match.group(0).strip()

            if matched_text:
                evidence.append(matched_text)

    # Preserve order while removing duplicates
    seen = set()
    unique_evidence = []

    for item in evidence:
        key = item.lower()

        if key not in seen:
            seen.add(key)
            unique_evidence.append(item)

    return unique_evidence

In [ ]:
def evaluate_lexical_indicator(text, indicator_id):
    rule = SEBRL_RULES[indicator_id]

    if rule["type"] != "lexical":
        raise ValueError(
            f"{indicator_id} is not a lexical indicator."
        )

    evidence = extract_pattern_evidence(
        text,
        rule["patterns"]
    )

    return {
        "supported": len(evidence) > 0,
        "evidence": evidence
    }

In [ ]:
if "Email" in SELECTED_BRANCHES:
    test_examples = [
        "Your account will be suspended. Verify immediately.",
        "Congratulations! You won a free prize. Claim it now.",
        "Please send your password and verification code.",
        "Meeting moved to Tuesday afternoon."
    ]

    for text in test_examples:

        print("\nTEXT:", text)

        for indicator_id, rule in SEBRL_RULES.items():

            if rule["type"] != "lexical":
                continue

            result = evaluate_lexical_indicator(
                text,
                indicator_id
            )

            if result["supported"]:
                print(
                    indicator_id,
                    "→",
                    result["evidence"]
                )


## 6. Selected evidence preparation
Email uses the existing subject/body rules and freeze gate, when selected.
PhiUSIIL and UCI remain technical U1: titles and engineered fields do not become
behavioral evidence. No absent Email frame or fabricated Email values are created.


In [ ]:
feature_frames = {name: df.copy() for name, df in feature_data.items()}
if "Email" in SELECTED_BRANCHES:
    email_features = feature_frames["Email"]
    email_features["evidence_text"] = (
        email_features["subject"].fillna("").str.cat(
            email_features["body"].fillna(""), sep=" ").map(normalize_match_text)
    )
if "PhiUSIIL" in SELECTED_BRANCHES:
    feature_frames["PhiUSIIL"]["evidence_text"] = ""


In [ ]:
for name, df in feature_frames.items():
    if "evidence_text" in df:
        print(name, "rows with non-empty evidence text:", df["evidence_text"].ne("").sum(), "/", len(df))


In [ ]:
LEXICAL_INDICATORS = [
    indicator_id
    for indicator_id, rule in SEBRL_RULES.items()
    if rule["type"] == "lexical"
]

print("Lexical indicators:", len(LEXICAL_INDICATORS))
print(LEXICAL_INDICATORS)

### 6.1 Lexical application boundary
Only the email branch may reach these English candidate rules. Unknown or
unsupported language remains unavailable. Historical rule definitions and
partial-availability aggregation are retained as drafts, not silently adopted.
A frozen, verified rule specification with original-field locators is required
before these routines can process research records (P 4.3.4/Tables 4.5-4.6).


In [ ]:
def apply_lexical_rules(df, text_column, prefix=""):
    df = df.copy()
    eligible = (
        df.get("language", pd.Series("und", index=df.index)).fillna("und").str.split("-").str[0].eq("en")
        & df[text_column].fillna("").str.strip().ne("")
    )
    for indicator_id in LEXICAL_INDICATORS:
        results = df.apply(
            lambda row: evaluate_lexical_indicator(row[text_column], indicator_id)
            if eligible.loc[row.name] else {"supported": False, "evidence": []},
            axis=1,
        )
        df[f"{prefix}{indicator_id}_supported"] = results.map(lambda x: x["supported"])
        df[f"{prefix}{indicator_id}_evidence"] = results.map(
            lambda x: json.dumps(x["evidence"], ensure_ascii=False))
    return df


In [ ]:
def prepare_email_branch(rows):
    # The existing scientific guard applies only to the selected Email component.
    if not FROZEN_RULE_SPECIFICATION_REFERENCE:
        raise RuntimeError("behavioral_specification_not_frozen: resolve qualifying/exclusion rules, original-field locators, language coverage and partial availability before research extraction.")
    return as_records(apply_lexical_rules(pd.DataFrame(rows), text_column="evidence_text"))

prepared_records = prepare_selected_branches(
    {name: as_records(df) for name, df in feature_frames.items()},
    SELECTED_BRANCHES, prepare_email=prepare_email_branch,
)
feature_frames = {name: pd.DataFrame(rows) for name, rows in prepared_records.items()}
if "Email" in SELECTED_BRANCHES:
    email_features = feature_frames["Email"]
if "PhiUSIIL" in SELECTED_BRANCHES:
    phiusiil_features = feature_frames["PhiUSIIL"]
if "UCI" in SELECTED_BRANCHES:
    uci_features = feature_frames["UCI"]


In [ ]:
# The shared dispatcher supplies unavailable technical indicators without Email callbacks.
for name in SELECTED_BRANCHES:
    if name != "Email":
        assert all(not feature_frames[name][f"{indicator}_available"].any()
                   for indicator in expected_indicators)


In [ ]:
support_counts = {
    name: {indicator: int(df[f"{indicator}_supported"].sum()) for indicator in LEXICAL_INDICATORS}
    for name, df in feature_frames.items()
}
for name, counts in support_counts.items():
    print(name, counts)


In [ ]:
# Retain the legacy combined Email/PhiUSIIL diagnostic schema.
summary_branches = tuple(n for n in SELECTED_BRANCHES if n in ("Email", "PhiUSIIL")) or SELECTED_BRANCHES
lexical_support_summary = pd.DataFrame({
    "indicator_id": LEXICAL_INDICATORS,
    **{f"{name.lower()}_supported": [support_counts[name][i] for i in LEXICAL_INDICATORS]
       for name in summary_branches},
})
lexical_support_summary


## 7. Identity and Structural Evidence Review

The remaining SE-BRL indicators `impersonation` and `brand_exploitation`
require identity-related evidence rather than simple lexical matching.

Before defining deterministic rules, the available email and webpage fields are
reviewed for evidence that can support:

- claimed identity;
- sender identity;
- domain identity;
- brand references;
- title/domain consistency; and
- webpage structural cues.

No identity-based indicator is assigned unless the available fields support a
reproducible rule.

In [ ]:
if "Email" in SELECTED_BRANCHES:
    email_identity_candidates = [
        col
        for col in email_features.columns
        if any(
            term in col.lower()
            for term in [
                "sender",
                "from",
                "reply",
                "url",
                "domain",
                "subject"
            ]
        )
    ]

    print("Email identity-related columns:")
    print(email_identity_candidates)


In [ ]:
if "PhiUSIIL" in SELECTED_BRANCHES:
    phiusiil_identity_candidates = [
        col
        for col in phiusiil_features.columns
        if any(
            term in col.lower()
            for term in [
                "url",
                "domain",
                "title",
                "brand",
                "login",
                "form",
                "password",
                "submit",
                "iframe",
                "redirect",
                "external",
                "link"
            ]
        )
    ]

    print("PhiUSIIL identity / structural columns:")
    print(phiusiil_identity_candidates)


In [ ]:
for name, df in feature_frames.items():
    if name == "Email":
        print(name, df[email_identity_candidates].dtypes)
    elif name == "PhiUSIIL":
        print(name, df[phiusiil_identity_candidates].dtypes)


In [ ]:
# Selected identity-field dtypes are shown in the preceding cell.


### 7.1 Identity and Structural Evidence Quality Check

Identity-related indicators require stronger evidence than the presence of a
brand or authority-related word.

The available fields are therefore inspected programmatically before defining
identity-based extraction rules.

This prevents unsupported assumptions such as treating every sender-domain
difference as impersonation or inventing thresholds for webpage similarity
scores without first understanding their source representation.

In [ ]:
if "Email" in SELECTED_BRANCHES:
    # Missing optional fields stay unknown; explicit supplied [] alone is known-empty.
    email_identity_quality = email_evidence_availability(as_records(email_features))
    email_identity_quality


In [ ]:
if "Email" in SELECTED_BRANCHES:
    # An absent optional column becomes an unknown diagnostic Series, never [].
    email_url_value_types = (
        email_features.get("urls", pd.Series(index=email_features.index, dtype="object"))
        .dropna().map(lambda x: type(x).__name__).value_counts()
    )
    email_url_value_types


In [ ]:
if "PhiUSIIL" in SELECTED_BRANCHES:
    phiusiil_identity_fields = [
        "DomainTitleMatchScore",
        "URLTitleMatchScore",
        "HasExternalFormSubmit",
        "HasSubmitButton",
        "HasPasswordField",
        "NoOfURLRedirect",
        "NoOfSelfRedirect",
        "NoOfiFrame",
        "NoOfExternalRef"
    ]

    # Optional diagnostics, never a U1 feature eligibility rule.
    phiusiil_identity_fields = [c for c in phiusiil_identity_fields if c in phiusiil_features]
    phiusiil_identity_quality = (
        phiusiil_features[
            phiusiil_identity_fields
        ]
        .agg(
            [
                "count",
                "min",
                "max",
                "mean"
            ]
        )
        .T
    ) if phiusiil_identity_fields else pd.DataFrame()

    phiusiil_identity_quality


In [ ]:
if "PhiUSIIL" in SELECTED_BRANCHES:
    for col in [
        "HasExternalFormSubmit",
        "HasSubmitButton",
        "HasPasswordField"
    ]:
        if col not in phiusiil_features:
            continue
        print(
            col,
            sorted(
                phiusiil_features[col]
                .dropna()
                .unique()
                .tolist()
            )
        )


In [ ]:
if "Email" in SELECTED_BRANCHES:
    email_identity_quality


In [ ]:
if "Email" in SELECTED_BRANCHES:
    email_url_value_types


In [ ]:
if "PhiUSIIL" in SELECTED_BRANCHES:
    phiusiil_identity_quality


### 7.2 Identity-Indicator Availability Decision

The available evidence does not provide a sufficiently direct and
source-independent basis for automatically asserting impersonation or brand
exploitation.

For email observations, sender information is available for some source
datasets, but no verified claimed-identity field is available for comparison.
The processed `urls` field is also numeric rather than a usable raw URL string.

For PhiUSIIL, domain/title similarity scores are available, but no arbitrary
threshold is introduced to convert those continuous scores into impersonation
or brand-exploitation labels.

Accordingly, `impersonation` and `brand_exploitation` are marked unavailable
unless a later source-supported rule can be established.

This does not make the entire Trust / Identity dimension unavailable because
other indicators in the dimension, including Authority and
Confidentiality / Isolation, remain directly assessable.

In [ ]:
for name, df in feature_frames.items():
    df["impersonation_available"] = False
    df["impersonation_supported"] = False
    df["impersonation_evidence"] = "[]"

    df["brand_exploitation_available"] = False
    df["brand_exploitation_supported"] = False
    df["brand_exploitation_evidence"] = "[]"


## 8. Historical structural interpretation (disabled)
The following source is preserved for provenance but is disabled: PhiUSIIL is
a technical U1 benchmark under the governing paper. HasPasswordField and related
technical fields do not become behavioral evidence in this path.


In [ ]:
# Historical branch retained for provenance; outside P Tables 4.1/4.7.
if False:
    phiusiil_features["call_to_action_lexical_supported"] = (
        phiusiil_features["call_to_action_supported"]
    )

    phiusiil_features["credential_sensitive_data_request_lexical_supported"] = (
        phiusiil_features["credential_sensitive_data_request_supported"]
    )


In [ ]:
# Historical branch retained for provenance; outside P Tables 4.1/4.7.
if False:
    def webpage_structural_evidence(row):
        evidence = []

        if row["HasSubmitButton"] == 1:
            evidence.append("HasSubmitButton=1")

        if row["HasExternalFormSubmit"] == 1:
            evidence.append("HasExternalFormSubmit=1")

        return evidence


    phiusiil_features["call_to_action_structural_evidence"] = (
        phiusiil_features.apply(
            webpage_structural_evidence,
            axis=1
        )
    )

    phiusiil_features["credential_structural_evidence"] = (
        phiusiil_features["HasPasswordField"]
        .map(
            lambda x:
                ["HasPasswordField=1"]
                if x == 1
                else []
        )
    )


In [ ]:
# Historical branch retained for provenance; outside P Tables 4.1/4.7.
if False:
    def combine_json_and_list(json_evidence, structural_evidence):
        lexical = json.loads(json_evidence)

        combined = lexical + structural_evidence

        # Preserve order and remove duplicates
        return list(dict.fromkeys(combined))


    phiusiil_features["call_to_action_evidence"] = [
        json.dumps(
            combine_json_and_list(
                lexical,
                structural
            ),
            ensure_ascii=False
        )
        for lexical, structural in zip(
            phiusiil_features["call_to_action_evidence"],
            phiusiil_features["call_to_action_structural_evidence"]
        )
    ]

    phiusiil_features["credential_sensitive_data_request_evidence"] = [
        json.dumps(
            combine_json_and_list(
                lexical,
                structural
            ),
            ensure_ascii=False
        )
        for lexical, structural in zip(
            phiusiil_features["credential_sensitive_data_request_evidence"],
            phiusiil_features["credential_structural_evidence"]
        )
    ]


In [ ]:
# Historical branch retained for provenance; outside P Tables 4.1/4.7.
if False:
    phiusiil_features["call_to_action_supported"] = (
        phiusiil_features[
            "call_to_action_lexical_supported"
        ]
        |
        phiusiil_features[
            "call_to_action_structural_evidence"
        ].map(bool)
    )

    phiusiil_features[
        "credential_sensitive_data_request_supported"
    ] = (
        phiusiil_features[
            "credential_sensitive_data_request_lexical_supported"
        ]
        |
        phiusiil_features[
            "credential_structural_evidence"
        ].map(bool)
    )


In [ ]:
# Historical branch retained for provenance; outside P Tables 4.1/4.7.
if False:
    print(
        "Call-to-Action before structural evidence:",
        phiusiil_features[
            "call_to_action_lexical_supported"
        ].sum()
    )

    print(
        "Call-to-Action after structural evidence:",
        phiusiil_features[
            "call_to_action_supported"
        ].sum()
    )

    print()

    print(
        "Credential request before structural evidence:",
        phiusiil_features[
            "credential_sensitive_data_request_lexical_supported"
        ].sum()
    )

    print(
        "Credential request after structural evidence:",
        phiusiil_features[
            "credential_sensitive_data_request_supported"
        ].sum()
    )


## 9. SE-BRL Dimension-State Construction

Indicator-level evidence is aggregated into the four SE-BRL behavioral
dimensions.

Each dimension receives one of three states:

- `Supported` — at least one available indicator in the dimension is supported;
- `Absent` — the dimension is assessable, but none of its available indicators
  is supported; or
- `Unavailable` — none of the indicators required to assess the dimension is
  available for the observation.

Unavailable indicators do not count as negative evidence.

This distinction ensures that missing modality evidence is not incorrectly
interpreted as behavioral absence.

In [ ]:
if "Email" in SELECTED_BRANCHES:
    email_text_available = (
        email_features["evidence_text"]
        .fillna("")
        .str.strip()
        .ne("")
    )

    email_text_available = email_text_available & (
        email_features.get("language", pd.Series("und", index=email_features.index))
        .fillna("und").str.split("-").str[0].eq("en")
    )
    for indicator_id in LEXICAL_INDICATORS:
        email_features[
            f"{indicator_id}_available"
        ] = email_text_available

    # Already established as unavailable
    email_features["impersonation_available"] = False
    email_features["brand_exploitation_available"] = False


In [ ]:
# Technical evidence availability is assigned uniformly in the next cell.


In [ ]:
# Technical availability is supplied by the dispatcher; it remains unavailable.
for name in SELECTED_BRANCHES:
    if name != "Email":
        for column, value in technical_indicator_columns(name).items():
            feature_frames[name][column] = value


In [ ]:
def derive_dimension_state(
    row,
    indicator_ids
):
    available = [
        bool(
            row[
                f"{indicator_id}_available"
            ]
        )
        for indicator_id in indicator_ids
    ]

    supported = [
        bool(
            row[
                f"{indicator_id}_supported"
            ]
        )
        for indicator_id in indicator_ids
    ]

    # Nothing in this dimension can be assessed
    if not any(available):
        return "Unavailable"

    # At least one available indicator has evidence
    if any(
        is_available and is_supported
        for is_available, is_supported
        in zip(available, supported)
    ):
        return "Supported"

    return "Absent"

In [ ]:
DIMENSION_INDICATORS = {
    dimension_id: list(
        dimension_info["indicators"].keys()
    )
    for dimension_id, dimension_info
    in SEBRL_CODEBOOK.items()
}

DIMENSION_INDICATORS

In [ ]:
for df in feature_frames.values():

    for dimension_id, indicator_ids in (
        DIMENSION_INDICATORS.items()
    ):

        df[
            f"{dimension_id}_state"
        ] = df.apply(
            lambda row:
                derive_dimension_state(
                    row,
                    indicator_ids
                ),
            axis=1
        )


In [ ]:
dimension_state_summary = []

for dataset_name, df in feature_frames.items():

    for dimension_id in DIMENSION_INDICATORS:

        counts = (
            df[
                f"{dimension_id}_state"
            ]
            .value_counts()
        )

        dimension_state_summary.append({
            "dataset": dataset_name,
            "dimension": dimension_id,
            "supported": int(
                counts.get("Supported", 0)
            ),
            "absent": int(
                counts.get("Absent", 0)
            ),
            "unavailable": int(
                counts.get("Unavailable", 0)
            )
        })

dimension_state_summary = pd.DataFrame(
    dimension_state_summary
)

dimension_state_summary


## 10. SE-BRL Behavioral Vector and Availability Mask

The categorical dimension states are converted into a numeric representation
for downstream machine-learning experiments.

For each SE-BRL dimension, two values are retained:

1. a behavioral support value:
   - `1` = Supported
   - `0` = Absent or Unavailable

2. an availability mask:
   - `1` = the dimension is assessable
   - `0` = the dimension is Unavailable

The availability mask prevents unavailable evidence from being interpreted as
behavioral absence.

For example:

- `support = 1, availability = 1` means Supported;
- `support = 0, availability = 1` means Absent; and
- `support = 0, availability = 0` means Unavailable.

This two-part representation preserves modality awareness while remaining
compatible with standard machine-learning models.

In [ ]:
def encode_dimension_state(state):
    if state == "Supported":
        return 1, 1

    if state == "Absent":
        return 0, 1

    if state == "Unavailable":
        return 0, 0

    raise ValueError(
        f"Unexpected SE-BRL state: {state}"
    )

In [ ]:
sebrl_datasets = feature_frames

for dataset_name, df in sebrl_datasets.items():

    for dimension_id in DIMENSION_INDICATORS:

        state_col = f"{dimension_id}_state"

        encoded = df[state_col].map(
            encode_dimension_state
        )

        df[f"{dimension_id}_supported"] = (
            encoded.map(lambda x: x[0])
        )

        df[f"{dimension_id}_available"] = (
            encoded.map(lambda x: x[1])
        )


In [ ]:
SEBRL_SUPPORT_COLUMNS = [
    f"{dimension_id}_supported"
    for dimension_id in DIMENSION_INDICATORS
]

SEBRL_AVAILABILITY_COLUMNS = [
    f"{dimension_id}_available"
    for dimension_id in DIMENSION_INDICATORS
]

print("Support vector:")
print(SEBRL_SUPPORT_COLUMNS)

print("\nAvailability mask:")
print(SEBRL_AVAILABILITY_COLUMNS)

In [ ]:
for dataset_name, df in sebrl_datasets.items():

    print(f"\n{dataset_name}")

    for dimension_id in DIMENSION_INDICATORS:

        state_col = f"{dimension_id}_state"
        support_col = f"{dimension_id}_supported"
        available_col = f"{dimension_id}_available"

        check = pd.crosstab(
            df[state_col],
            [
                df[support_col],
                df[available_col]
            ]
        )

        print(f"\n{dimension_id}")
        print(check)

In [ ]:
for dataset_name, df in sebrl_datasets.items():

    for dimension_id in DIMENSION_INDICATORS:

        state_col = f"{dimension_id}_state"
        support_col = f"{dimension_id}_supported"
        available_col = f"{dimension_id}_available"

        assert (
            df.loc[
                df[state_col] == "Supported",
                support_col
            ]
            == 1
        ).all()

        assert (
            df.loc[
                df[state_col] == "Supported",
                available_col
            ]
            == 1
        ).all()

        assert (
            df.loc[
                df[state_col] == "Absent",
                support_col
            ]
            == 0
        ).all()

        assert (
            df.loc[
                df[state_col] == "Absent",
                available_col
            ]
            == 1
        ).all()

        assert (
            df.loc[
                df[state_col] == "Unavailable",
                support_col
            ]
            == 0
        ).all()

        assert (
            df.loc[
                df[state_col] == "Unavailable",
                available_col
            ]
            == 0
        ).all()

print(
    "All SE-BRL dimension states encoded correctly."
)

In [ ]:
vector_summary = []

for dataset_name, df in sebrl_datasets.items():

    for dimension_id in DIMENSION_INDICATORS:

        vector_summary.append({
            "dataset": dataset_name,
            "dimension": dimension_id,
            "support_rate_percent": round(
                df[
                    f"{dimension_id}_supported"
                ].mean() * 100,
                2
            ),
            "availability_rate_percent": round(
                df[
                    f"{dimension_id}_available"
                ].mean() * 100,
                2
            )
        })

vector_summary = pd.DataFrame(
    vector_summary
)

vector_summary

## 11. Indicator-Level Evidence Representation

The compact SE-BRL model vector represents behavioral support at the
four-dimension level.

Indicator-level outputs are retained separately for explainability and
validation.

For each of the twelve indicators, the exported evidence representation
contains:

- indicator availability;
- indicator support status; and
- exact automatically extracted evidence.

Keeping the indicator evidence separate from the compact model vector allows
downstream validation of evidence localization without introducing evidence
strings directly into the classifier input.

In [ ]:
SEBRL_INDICATORS = (
    sebrl_codebook_df["indicator_id"]
    .tolist()
)

print("Indicators:", len(SEBRL_INDICATORS))
print(SEBRL_INDICATORS)

In [ ]:
def build_indicator_evidence_table(df, dataset_name):
    rows = []

    for indicator_id in SEBRL_INDICATORS:

        availability_col = f"{indicator_id}_available"
        support_col = f"{indicator_id}_supported"
        evidence_col = f"{indicator_id}_evidence"

        temp = pd.DataFrame({
            "observation_id": df["observation_id"],
            "dataset": dataset_name,
            "indicator_id": indicator_id,
            "available": df[availability_col],
            "supported": df[support_col],
            "evidence": df[evidence_col],
        })

        rows.append(temp)

    return pd.concat(
        rows,
        ignore_index=True
    )

In [ ]:
indicator_evidence_by_branch = {
    name: build_indicator_evidence_table(df, name)
    for name, df in feature_frames.items()
}


In [ ]:
indicator_evidence = pd.concat(list(indicator_evidence_by_branch.values()), ignore_index=True)
print("Indicator evidence rows:", len(indicator_evidence))
indicator_evidence.head()


In [ ]:
invalid_unavailable_support = (
    indicator_evidence[
        (~indicator_evidence["available"])
        &
        (indicator_evidence["supported"])
    ]
)

print(
    "Unavailable-but-supported records:",
    len(invalid_unavailable_support)
)

In [ ]:
indicator_evidence_counts = (
    indicator_evidence
    .groupby("dataset")
    .size()
)

indicator_evidence_counts

In [ ]:
assert len(indicator_evidence) == sum(len(df) for df in feature_frames.values()) * 12
assert len(invalid_unavailable_support) == 0
print("All selected indicator-level evidence integrity checks passed.")


## 12. Compact SE-BRL Model Feature Tables

The final SE-BRL model representation contains only the metadata required to
preserve the experimental design together with the compact behavioral vector.

For each observation, the exported model table contains:

- observation identifier;
- target;
- leakage-prevention group identifier;
- development/calibration/test partition;
- development cross-validation fold;
- four SE-BRL dimension support values; and
- four modality-availability mask values.

Indicator evidence strings remain stored separately and are not included as
direct classifier inputs.

In [ ]:
BASE_MODEL_COLUMNS = [
    "observation_id",
    "target",
    "group_id",
    "partition",
    "cv_fold",
]

SEBRL_MODEL_COLUMNS = (
    BASE_MODEL_COLUMNS
    + SEBRL_SUPPORT_COLUMNS
    + SEBRL_AVAILABILITY_COLUMNS
)

SEBRL_MODEL_COLUMNS

In [ ]:
compact_features = {
    name: df[SEBRL_MODEL_COLUMNS].copy() for name, df in feature_frames.items()
}


In [ ]:
for name, df in compact_features.items():

    print(
        f"{name}: "
        f"{len(df):,} rows × {len(df.columns)} columns"
    )


In [ ]:
for name, df in compact_features.items():

    assert df["observation_id"].is_unique
    assert df["partition"].notna().all()

    assert (
        df.loc[
            df["partition"] == "development",
            "cv_fold"
        ]
        .notna()
        .all()
    )

    assert (
        df.loc[
            df["partition"] != "development",
            "cv_fold"
        ]
        .isna()
        .all()
    )

print(
    "All compact SE-BRL feature tables "
    "preserve preprocessing metadata."
)

# Exact metadata/coverage checks, beyond uniqueness and non-null counts.
for name, df in compact_features.items():
    expected = feature_data[name][BASE_MODEL_COLUMNS].set_index("observation_id").sort_index()
    actual = df[BASE_MODEL_COLUMNS].set_index("observation_id").sort_index()
    assert actual.equals(expected), f"{name}: feature metadata changed"


In [ ]:
for name, df in compact_features.items():

    print(f"\n{name}")

    print(
        "Support values:",
        sorted(
            pd.unique(
                df[
                    SEBRL_SUPPORT_COLUMNS
                ].values.ravel()
            )
        )
    )

    print(
        "Availability values:",
        sorted(
            pd.unique(
                df[
                    SEBRL_AVAILABILITY_COLUMNS
                ].values.ravel()
            )
        )
    )


## 13. Export SE-BRL Feature Artifacts

The finalized SE-BRL outputs are exported in separate representations for
modeling, validation, and documentation.

The compact model tables contain only the four dimension-level support values
and four availability-mask values together with immutable experimental
metadata.

Indicator-level evidence is exported separately so that explanation and
validation can be performed without exposing evidence strings directly to the
classifiers.

The canonical SE-BRL codebook and summary tables are also exported for
reproducibility.

In [ ]:
# Prepare all bytes before any write. Only selected branch tables are exported.
shared_tables = (
    indicator_evidence, sebrl_codebook_df, evidence_availability_df,
    lexical_support_summary, dimension_state_summary, vector_summary,
)
shared_output_bytes = {
    name: table.to_csv(index=False).encode("utf-8")
    for name, table in zip(SHARED_OUTPUTS, shared_tables)
}


In [ ]:
exported_paths = export_selected_handoff(
    RUN_ROOT, SELECTED_BRANCHES,
    {name: as_records(df) for name, df in feature_data.items()},
    {name: as_records(df) for name, df in compact_features.items()},
    shared_outputs=shared_output_bytes,
)
for path in exported_paths:
    print("Created:", path.relative_to(REPO_ROOT))


In [ ]:
# Indicator evidence was included in the validated exclusive export above.
print("Selected indicator evidence rows:", len(indicator_evidence))


In [ ]:
# Shared codebook/availability/summaries were exported in the same protected set.


In [ ]:
for path in exported_paths:
    print("OK" if path.is_file() else "MISSING", path.relative_to(REPO_ROOT))


## 14. Feature Extraction Summary and Documentation

### Completed Feature Extraction Tasks

The SE-BRL feature-extraction stage produced a deterministic,
modality-aware behavioral representation from the fixed preprocessing outputs
generated by `03_preprocessing.ipynb`.

The following tasks were completed:

- loaded the fixed preprocessing datasets and verified immutable partition
  metadata;
- defined the canonical four-dimension, twelve-indicator SE-BRL codebook;
- established modality-specific evidence availability rules;
- implemented deterministic lexical indicator extraction;
- preserved exact automatically matched evidence for explainability;
- incorporated deterministic webpage structural evidence from PhiUSIIL;
- explicitly separated unsupported identity inference from unavailable
  evidence;
- constructed dimension-level `Supported`, `Absent`, and `Unavailable` states;
- encoded the four-dimensional SE-BRL support vector and modality availability
  mask;
- constructed indicator-level evidence records for downstream validation; and
- exported compact SE-BRL model features and supporting documentation artifacts.

### Canonical SE-BRL Representation

The behavioral representation contains four dimensions:

1. Pressure / Threat
2. Lure / Attention
3. Trust / Identity
4. Requested Action / Consequence

These dimensions are derived from twelve subordinate indicators.

Each dimension is represented using:

- a support value (`1` = Supported, `0` otherwise); and
- an availability value (`1` = assessable, `0` = Unavailable).

This preserves the distinction between behavioral absence and unavailable
modality evidence.

### Email Feature Extraction

Email observations use available subject and body content for lexical
behavioral extraction.

The following dimension support rates were observed:

- Pressure / Threat: approximately 10.26%;
- Lure / Attention: approximately 13.97%;
- Trust / Identity: approximately 16.59%; and
- Requested Action / Consequence: approximately 35.53%.

The email modality remains fully assessable for the four dimension-level
representations under the current extraction rules.

### PhiUSIIL Feature Extraction

PhiUSIIL lexical extraction uses the available webpage title field.

Explicit webpage-structure fields were additionally used as deterministic
evidence:

- `HasSubmitButton = 1`;
- `HasExternalFormSubmit = 1`; and
- `HasPasswordField = 1`.

These structural fields substantially increased detection of requested-action
behaviors compared with title-only lexical extraction.

The final Requested Action / Consequence support rate is approximately 43.22%.

No arbitrary threshold was introduced for continuous domain/title similarity
scores.

### Identity-Based Indicators

`Impersonation` and `Brand Exploitation` were not inferred using arbitrary
sender or domain mismatch rules.

The available datasets do not provide a sufficiently direct and
source-independent basis for asserting these indicators automatically under the
current extraction design.

They are therefore treated as unavailable where a defensible deterministic
rule cannot be established.

Other indicators within the Trust / Identity dimension remain assessable.

### URL-Only Datasets

UCI and ISCX contain engineered URL or webpage-characteristic features without
the underlying natural-language or page-content evidence required for direct
SE-BRL behavioral inference.

Accordingly, their behavioral dimensions are encoded as:

- support = `0`; and
- availability = `0`.

This represents `Unavailable`, not behavioral absence.

### Evidence Representation

Indicator-level evidence is stored separately from the compact classifier
inputs.

Each indicator evidence record preserves:

- observation identifier;
- dataset;
- indicator identifier;
- availability;
- support status; and
- automatically extracted evidence.

The combined indicator evidence table contains 3,542,712 records.

No unavailable indicator was simultaneously marked as supported.

### Generated Artifacts

Compact SE-BRL model feature tables:

`experiments/data/processed/features/sebrl/`

Indicator-level evidence:

`experiments/results/feature_extraction/evidence/`

Canonical codebook and evidence-availability documentation:

`experiments/results/feature_extraction/`

Feature-extraction summary tables:

`experiments/results/feature_extraction/summaries/`

### Next Stage

The next stage is `05_experiments.ipynb`.

That notebook will use the fixed development, calibration, test, and
cross-validation assignments together with the SE-BRL feature representations
to execute the controlled machine-learning experiments.

Learned preprocessing, classifier fitting, threshold calibration, and final
test evaluation must continue to respect the leakage controls established in
the preceding stages.